## Path setup and load everything

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

import config
import models
import attacks
import evaluation

import numpy as np
import torch
import joblib

# Load the preprocessed arrays and fitted objects from Stage 2.
data = np.load(config.PROCESSED_DIR / "stage2_arrays.npz")
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]
label_encoder = joblib.load(config.PROCESSED_DIR / "label_encoder.joblib")
class_names = list(label_encoder.classes_)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loaded. X_test: {X_test.shape}  device: {device}")

Loaded. X_test: (718, 9)  device: cuda


## Retrain the baseline CNN

In [2]:
from sklearn.utils.class_weight import compute_class_weight

# Class weights as same as stage 3
w = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(w, dtype=torch.float32, device=device)

# Train the baseline CNN on the full duplicated training set
cnn = models.CNN1D(n_features=9, n_classes=6)
cnn = models.train_cnn(
    cnn, X_train, y_train,
    n_epochs=50, device=device,
    class_weights=class_weights,
    random_seed=config.RANDOM_SEED,
)

print("Baseline CNN trained.")

    epoch   1/50     loss 1.5270
    epoch   5/50     loss 0.0755
    epoch  10/50     loss 0.0063
    epoch  15/50     loss 0.0027
    epoch  20/50     loss 0.0021
    epoch  25/50     loss 0.0016
    epoch  30/50     loss 0.0013
    epoch  35/50     loss 0.0011
    epoch  40/50     loss 0.0010
    epoch  45/50     loss 0.0009
    epoch  50/50     loss 0.0009
Baseline CNN trained.


## Wrap CNN and generate FGSM examples

In [3]:
# Wrap the trained CNN
classifier = attacks.wrap_cnn_for_art(cnn, n_features=9, n_classes=6, device=device)

# Baseline: how does the CNN do on the clean test set
cnn.eval()
with torch.no_grad():
    clean_logits = cnn(torch.tensor(X_test, dtype=torch.float32, device=device))
    clean_pred = clean_logits.argmax(dim=1).cpu().numpy()

from sklearn.metrics import f1_score
clean_f1 = f1_score(y_test, clean_pred, average="macro", zero_division=0)
print(f"CNN clean macro-F1 (reference): {clean_f1:.4f}\n")

# Generate FGSM adversarial examples from the test set at a moderate epsilon
X_adv_fgsm = attacks.generate_fgsm(classifier, X_test, epsilon=0.10)

# How does the CNN do on the purturbated test set
with torch.no_grad():
    adv_logits = cnn(torch.tensor(X_adv_fgsm, dtype=torch.float32, device=device))
    adv_pred = adv_logits.argmax(dim=1).cpu().numpy()

adv_f1 = f1_score(y_test, adv_pred, average="macro", zero_division=0)
print(f"CNN FGSM macro-F1 (epsilon=0.10): {adv_f1:.4f}")
print(f"Degredation: {clean_f1 - adv_f1:.4f} drop")

CNN clean macro-F1 (reference): 0.6588

CNN FGSM macro-F1 (epsilon=0.10): 0.1427
Degredation: 0.5160 drop


## Epsilon sweep, FGSM and PGD, against both models

In [5]:
from sklearn.metrics import f1_score

# Helper: get macro-F1 for a model's predictions on some inputs
def cnn_macro_f1(X):
    cnn.eval()
    with torch.no_grad():
        pred = cnn(torch.tensor(X, dtype=torch.float32, device=device)).argmax(dim=1).cpu().numpy()
    return f1_score(y_test, pred, average="macro", zero_division=0)

def rf_macro_f1(X):
    return f1_score(y_test, rf.predict(X), average="macro", zero_division=0)

# Retrain the RF so the notebook is self-contained
rf = models.build_random_forest(random_seed=config.RANDOM_SEED)
rf.fit(X_train, y_train)

# Clean baselines
clean_cnn = cnn_macro_f1(X_test)
clean_rf = rf_macro_f1(X_test)
print(f"CLEAN   CNN={clean_cnn:.4f}     RF={clean_rf:.4f}\n")

# Sweep over the epsilon list from config for both attacks
epsilons = config.FGSM_EPSILONS

results = {"eps": [], "fgsm_cnn": [], "fgsm_rf": [], "pgd_cnn": [], "pgd_rf": []}

for eps in epsilons:
    # FGSM at this epsilon crafted on the cnn
    Xf = attacks.generate_fgsm(classifier, X_test, epsilon=eps)
    # PGD at this epsilon crafted on the cnn
    Xp = attacks.generate_pgd(classifier, X_test, epsilon=eps)

    # Feed same cradfted frames to both models (transfer attack)
    results["eps"].append(eps)
    results["fgsm_cnn"].append(cnn_macro_f1(Xf))
    results["fgsm_rf"].append(rf_macro_f1(Xf))
    results["pgd_cnn"].append(cnn_macro_f1(Xp))
    results["pgd_rf"].append(rf_macro_f1(Xp))

    print(f"eps={eps:.2f} | FGSM: CNN={results['fgsm_cnn'][-1]:.4f} RF={results['fgsm_rf'][-1]:.4f}"
          f" | PGD: CNN={results['pgd_cnn'][-1]:.4f} RF={results['pgd_rf'][-1]:.4f}")

CLEAN   CNN=0.6588     RF=0.7761



PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.01 | FGSM: CNN=0.6588 RF=0.4954 | PGD: CNN=0.6588 RF=0.3286


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.05 | FGSM: CNN=0.3929 RF=0.1468 | PGD: CNN=0.4190 RF=0.1493


...

eps=0.10 | FGSM: CNN=0.1427 RF=0.1547 | PGD: CNN=0.1635 RF=0.1509


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.20 | FGSM: CNN=0.1398 RF=0.1656 | PGD: CNN=0.1416 RF=0.1517


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.30 | FGSM: CNN=0.1608 RF=0.1656 | PGD: CNN=0.1402 RF=0.1509


## Verify the perturbation and break down per-class

In [10]:
from sklearn.metrics import classification_report

# Part 1: is the perturbation real at eps=0.01
X_adv_001 = attacks.generate_fgsm(classifier, X_test, epsilon=0.01)

# How much did the inputs actually change?
diff = np.abs(X_adv_001 - X_test)
print("PERTURBATION CHECK at eps=0.01")
print(f"    max abs change per feature  : {diff.max():.5f}")
print(f"    mean abs change             : {diff.mean():.5f}")
print(f"    rows that changed at all    : {(diff.sum(axis=1) > 0).sum()} / {len(X_test)}")

# Did the CNN predictions actually change?
cnn.eval()
with torch.no_grad():
    p_clean = cnn(torch.tensor(X_test, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    p_adv = cnn(torch.tensor(X_adv_001, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
print(f"    CNN predictions changed     : {(p_clean != p_adv).sum()} / {len(X_test)}")

# Part 2: per-class breakdown, RF under FGSM eps=0.01
print("\nRF per-class CLEAN:")
print(classification_report(y_test, rf.predict(X_test), target_names=class_names, zero_division=0))
print("\nRF per-class under FGSM eps=0.01:")
print(classification_report(y_test, rf.predict(X_adv_001), target_names=class_names, zero_division=0))

PERTURBATION CHECK at eps=0.01
    max abs change per feature  : 0.01000
    mean abs change             : 0.00870
    rows that changed at all    : 718 / 718
    CNN predictions changed     : 0 / 718

RF per-class CLEAN:
                         precision    recall  f1-score   support

                    DoS       1.00      0.75      0.86         4
                 benign       1.00      1.00      1.00       709
           spoofing-GAS       0.00      0.00      0.00         1
           spoofing-RPM       0.67      1.00      0.80         2
         spoofing-SPEED       1.00      1.00      1.00         1
spoofing-STEERING_WHEEL       1.00      1.00      1.00         1

               accuracy                           1.00       718
              macro avg       0.78      0.79      0.78       718
           weighted avg       1.00      1.00      1.00       718


RF per-class under FGSM eps=0.01:
                         precision    recall  f1-score   support

                    DoS 